In [7]:
import tiktoken

enc = tiktoken.get_encoding("o200k_base")
print("Vocabulary size:", enc.n_vocab)

Vocabulary size: 200019


In [8]:
text = "Hello world! Tokens are fun."
ids = enc.encode(text)

print("Token IDs:", ids)
print("Pieces:   ", [enc.decode([i]) for i in ids])
print("Count:    ", len(ids))

Token IDs: [13225, 2375, 0, 124325, 553, 2827, 13]
Pieces:    ['Hello', ' world', '!', ' Tokens', ' are', ' fun', '.']
Count:     7


In [9]:
prompts = [
    "What is a token?",
    "Summarize this email in one line: Hi team, the meeting moved to 4 pm on Friday.",
    "Write a Python function that checks if a number is prime.",
    "नमस्ते, आप कैसे हैं? मुझे AI इंजीनियर बनना है।",
    "Hello, how are you? I want to become an AI engineer.",
]

for p in prompts:
    n_tokens = len(enc.encode(p))
    print(f"{n_tokens:>3} tokens | {len(p):>3} chars | {p[:50]}")

  5 tokens |  16 chars | What is a token?
 22 tokens |  79 chars | Summarize this email in one line: Hi team, the mee
 12 tokens |  57 chars | Write a Python function that checks if a number is
 18 tokens |  46 chars | नमस्ते, आप कैसे हैं? मुझे AI इंजीनियर बनना है।
 14 tokens |  52 chars | Hello, how are you? I want to become an AI enginee


In [10]:
# USD per 1 million tokens (OpenAI pricing page, Nov 2026, check for updates)
PRICE_IN = 0.15    # gpt-4o-mini input
PRICE_OUT = 0.60   # gpt-4o-mini output
EXPECTED_OUTPUT_TOKENS = 100   # assume each answer is about 100 tokens
USD_TO_INR = 84    # rough rate, for a feel in rupees

for p in prompts:
    n_in = len(enc.encode(p))
    cost = (n_in * PRICE_IN + EXPECTED_OUTPUT_TOKENS * PRICE_OUT) / 1_000_000
    print(f"{n_in:>3} in + {EXPECTED_OUTPUT_TOKENS} out = ${cost:.6f}  (Rs {cost * USD_TO_INR:.4f})")

per_call = (20 * PRICE_IN + 100 * PRICE_OUT) / 1_000_000
print(f"\n1 lakh calls/day of a 20-token prompt: ${per_call * 100_000:.2f} per day")

  5 in + 100 out = $0.000061  (Rs 0.0051)
 22 in + 100 out = $0.000063  (Rs 0.0053)
 12 in + 100 out = $0.000062  (Rs 0.0052)
 18 in + 100 out = $0.000063  (Rs 0.0053)
 14 in + 100 out = $0.000062  (Rs 0.0052)

1 lakh calls/day of a 20-token prompt: $6.30 per day


In [11]:
old = tiktoken.get_encoding("cl100k_base")   # older tokenizer (GPT-4, GPT-3.5)

hindi = prompts[3]
english = prompts[4]

for name, e in [("old cl100k", old), ("new o200k", enc)]:
    h = len(e.encode(hindi))
    en = len(e.encode(english))
    print(f"{name:>10}: Hindi {h} tokens, English {en} tokens, Hindi/English = {h / en:.1f}x")

old cl100k: Hindi 46 tokens, English 14 tokens, Hindi/English = 3.3x
 new o200k: Hindi 18 tokens, English 14 tokens, Hindi/English = 1.3x


## What I found

- English uses about 3.7 characters per token, close to the "4 characters per token" rule.
- Same sentence: Hindi 18 tokens vs English 14 tokens with o200k (1.3x).
- With the older cl100k tokenizer, Hindi needed 46 tokens (3.3x English). Newer tokenizers handle Hindi much better.
- Output tokens cost 4x more than input tokens for gpt-4o-mini, so most of the cost of a short prompt comes from the answer.
- One call costs a tiny fraction of a rupee, but cost adds up fast at scale.

# Tokens and Cost Notebook
Counting tokens and estimating API cost for 5 prompts, including English vs Hindi.